In [1]:
import pandas as pd
import numpy as np
import os
import seaborn as sns
GTE_DIR=os.environ["GTE_DIR"]
from glaciation_time_estimator.auxiliary_func.config_reader import read_config
from glaciation_time_estimator.auxiliary_func.Helper_fun import float_range
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib as mpl
import matplotlib.gridspec as gridspec
import cmcrameri.cm
from glaciation_time_estimator.auxiliary_func.chunking_data import ChunkLoader
import datetime as dt

In [ ]:
colormap = cmcrameri.cm.lipariS

In [ ]:
glac_threshold=0.4
def calculate_pole_temp_occurance_rate(dfs):
    cloud_chunk , _ , _ = dfs
    return pd.pivot_table(cloud_chunk,values='is_glaciating' ,index=["Hemisphere"], columns=["min_temp"], aggfunc='mean')
def calculate_pole_temp_occurance_rate_glac_possible(dfs):
    cloud_chunk , _ , _ = dfs
    return pd.pivot_table(cloud_chunk[cloud_chunk.max_water_frac > glac_threshold],values='is_glaciating' ,index=["Hemisphere"], columns=["min_temp"], aggfunc='mean')


# 1 Load data

CLAAS data (original - 0.15)

In [ ]:
config_claas_low_res = read_config("/wolke_scratch/dnikolo/Glaciation_time_estimator/configs/config_n2o.yaml")

In [ ]:
config_claas_high_res = read_config("/wolke_scratch/dnikolo/Glaciation_time_estimator/configs/config_n2o.yaml")
loader_claas_high_res = ChunkLoader([2009],config_claas_high_res,filter=True)

Cloud_fps: {2009: '/wolke_scratch/dnikolo/GTE_Results/Final_results/2009_all.parquet'}


KeyboardInterrupt: 

In [ ]:
config_icon = read_config("/wolke_scratch/dnikolo/Glaciation_time_estimator/configs/ICON_output/nudged_3y/n2o_read.yaml")

In [ ]:
loader_icon = ChunkLoader([2009],config_icon,filter=True)

Model 160 km

In [ ]:
clouds_model = loader_icon.cloud_chunk
glac_model = loader_icon.glac_chunk
clouds_high_res = loader_claas_high_res.cloud_chunk
glac_high_res = loader_claas_high_res.glac_chunk


In [ ]:
start_time=pd.Timestamp("2008-09-01")
end_time= pd.Timestamp("2008-09-08")
def time_filter(df):
    return df[(df.track_start_time>=start_time) & (df.track_start_time<end_time)]
def get_peak_if(glac_df_row):
    return glac_df_row["ice_frac_hist"][glac_df_row["Glac_peak_ind"]]
def set_peak_if(glac_df):
    glac_df["Peak IF"] = glac_df.apply(get_peak_if,axis=1)
def filter_df(df):
    return df[(~df.is_large_pix_cloud) & (abs(df.avg_lat)>30)]
def preproc_glac(glac_df):
    set_peak_if(glac_df)
    return filter_df(time_filter(glac_df))
def preproc_cloud(cloud_df):
    return filter_df(time_filter(cloud_df))

set_peak_if(glac_high_res)
set_peak_if(glac_model)
glac_low_res = preproc_glac(glac_low_res)
clouds_low_res = preproc_cloud(clouds_low_res)
# set_peak_if(glac_model)
# set_peak_if(glac_high_res)
# set_peak_if(glac_low_res)
# glac_high_res = time_filter(loader.glac_chunk)
# clouds_low_res=time_filter(clouds_low_res)
# glac_low_res=time_filter(glac_low_res)

# clouds_high_res = clouds_high_res[clouds_high_res["avg_size[km]"]>3e4]

In [ ]:
def set_is_glaciating(cloud_df,glac_df):
    cloud_df['is_glaciating'] = (
                    cloud_df.index.isin(
                        glac_df['Cloud_ID']
                    )
                )

set_is_glaciating(clouds_low_res,glac_low_res)
# set_is_glaciating(clouds_model_offline,glac_model_offline)
set_is_glaciating(clouds_model,glac_model)


In [ ]:
def set_hemisphere(cloud_df):
    cloud_df['Hemisphere'] = np.where(cloud_df['avg_lat']>=0,'North','South')
# set_hemisphere(clouds_model_offline)
set_hemisphere(clouds_model)